# Fragmenty wskazań bez zdarzeń

Jak zachowują się wskazania wagi silosu, kiedy nic się nie dzieje: szum, krótkie zakłócenia i wolne zmiany od czynników zewnętrznych. Zbiór: baza `2026-10-18 obiekt`, sesja 9, wariant `CURRENT_VARIANT`; obliczenia w `dms_analysis.between_events_fragments`.

Kroki:

1. **Fragmenty bez zdarzeń** — wskazania poza wszystkimi zdarzeniami (odsypy i zasypy, przyjęte i odrzucone) z marginesem `MARGIN` przed początkiem i po końcu; fragment kończy się też na przerwie w danych dłuższej niż `MAX_SAMPLE_GAP`; zostają fragmenty nie krótsze niż `MIN_DURATION` (odpadają krótkie przerwy w seriach odsypów, mieści się w nich najdłuższy odsyp z zapasem).
2. **Zakłócenia** — próbka odbiegająca od mediany z `SPIKE_WINDOW_SAMPLES` sąsiednich próbek o więcej niż `SPIKE_THRESHOLD_KG` (szukane w całej sesji, także przy zdarzeniach). Okno ok. 2 min, bo w sesji 9 są spadki wskazań o ok. 7 t trwające do 28 s; zakłócenie wykrywa się tylko, gdy trwa krócej niż połowa okna. Stały próg, bo przy rozdzielczości 20 kg i stałych wskazaniach typowy rozrzut wokół mediany często wynosi 0. Zakłócone próbki są pomijane w krokach 3 i 4.
3. **Nachylenie i odchylenie w oknach** `WINDOWS_S` — w każdym oknie (wyśrodkowanym na próbce) prosta dopasowana metodą najmniejszych kwadratów: nachylenie = szybkość zmian [kg/h], odchylenie standardowe wskazań od prostej = szum po odjęciu trendu [kg]. Okno liczone w próbkach (ok. 1 próbka na sekundę), tylko w całości wewnątrz fragmentu i bez zakłóceń.
4. **Odchylenie Allana** — zmienność średnich z kolejnych odcinków długości τ: σ(τ) = √(½ · średnia (ȳ_{k+1} − ȳ_k)²). Spada z τ tam, gdzie dominuje szum (uśrednianie pomaga), rośnie tam, gdzie dominuje dryft; minimum = najlepszy czas uśredniania.

Uwaga do interpretacji krótkich okien: przy rozdzielczości 20 kg jeden krok wskazania w oknie 5 s to nachylenie ok. 14 000 kg/h, a stałe wskazania dają odchylenie 0 — w krótkich oknach nachylenie mierzy głównie zaokrąglenie miernika, nie zmiany masy.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from IPython.display import display

from dms_analysis.between_events_fragments import (
    DAY_HOURS,
    NIGHT_HOURS,
    FragmentBrowser,
    allan_deviation,
    allan_figure,
    fragment_allan,
    fragment_allan_figure,
    fragment_table,
    fragment_windows,
    prepare_samples,
    spike_episodes,
    spike_windows,
    window_box_figure,
    window_summary,
    white_noise_samples,
)
from dms_analysis.prepared import load_session
from dms_analysis.signals import CURRENT_VARIANT
from dms_analysis.snapshot import ANALYSIS_SESSION, ANALYSIS_SOURCE
from dms_analysis.tables import collapsible

DB_PATH = Path("../data/prepared/obiekt-sesja-9.db")
FIGURE_WIDTH = 1200
SPIKE_PADDING = pd.Timedelta(seconds=60)

pd.set_option("display.max_rows", 500)

In [ ]:
session = load_session(DB_PATH, ANALYSIS_SOURCE, ANALYSIS_SESSION, CURRENT_VARIANT)
samples = prepare_samples(session)
fragments = fragment_table(samples)

print(f"zdarzeń: {len(session.events)}, fragmentów: {len(fragments)}, łącznie {fragments['duration_min'].sum() / 60:.1f} h")
display(collapsible(fragments.round(1), "Fragmenty bez zdarzeń"))

## Przegląd fragmentów

Wskazania (z medianą do wykrywania zakłóceń i zakłóconymi próbkami na czerwono), pod nimi nachylenie i odchylenie w wybranym oknie dla tych samych chwil. Zdarzenia w marginesie podglądu: zielone przyjęte, czerwone odrzucone. Oś wskazań bez zakłóconych próbek (wychodzą poza wykres). Kolejność: `ORDER` (domyślnie od największej zmiany na godzinę).

In [ ]:
ORDER = fragments["change_kg_h"].abs().sort_values(ascending=False).index

FragmentBrowser(samples, session.events, fragment_windows(fragments.loc[ORDER]), "Fragmenty bez zdarzeń", FIGURE_WIDTH).widget

## Zakłócenia

Ciągi zakłóconych próbek w całej sesji: miejsce (we fragmencie, w zdarzeniu albo jego marginesie, poza fragmentem) i największa odchyłka od mediany.

In [ ]:
episodes = spike_episodes(samples)
display(episodes)

FragmentBrowser(samples, session.events, spike_windows(episodes, SPIKE_PADDING), "Zakłócenia", FIGURE_WIDTH).widget

## Nachylenie i odchylenie w oknach różnej długości

Rozkład ze wszystkich okien we fragmentach (okno przy każdej próbce, więc okna się nakładają). Nachylenie jako wartość bezwzględna.

In [ ]:
display(window_summary(samples).round(1))

window_box_figure(samples, "std_kg", "Odchylenie od prostej w oknie", "odchylenie [kg]", FIGURE_WIDTH).show()
window_box_figure(samples, "slope_kg_h", "|Nachylenie| prostej w oknie", "|nachylenie| [kg/h]", FIGURE_WIDTH).show()

## Odchylenie Allana

Wszystkie fragmenty razem (pary średnich tylko wewnątrz jednego fragmentu; dla dużych τ liczą się tylko długie fragmenty, liczba par w tabeli). Do porównania:

- **biały szum** — w miejsce wskazań we fragmentach niezależne losowe wartości z rozkładu normalnego o odchyleniu równym σ(1 s) wskazań, zaokrąglone do działki `RESOLUTION_KG`, z tymi samymi fragmentami i pominiętymi zakłóceniami; tak wyglądałaby krzywa, gdyby wskazania różniły się tylko szumem odczytu (spadek jak 1/√τ),
- **noc i dzień** — tylko próbki z tych godzin według zegara RPi, który według użytkownika spieszy się o ok. 4 h (13-20 na zegarze to ok. 9-16 rzeczywistej; pary średnich nie przechodzą przez granicę przedziału godzin).

In [ ]:
allan = allan_deviation(samples)
white = allan_deviation(white_noise_samples(samples, allan["allan_kg"].iloc[0]))
night = allan_deviation(samples, NIGHT_HOURS)
afternoon = allan_deviation(samples, DAY_HOURS)

display(
    pd.DataFrame(
        {
            "tau_s": allan["tau_s"],
            "wszystkie": allan["allan_kg"],
            "noc": night["allan_kg"],
            "dzień": afternoon["allan_kg"],
            "biały szum": white["allan_kg"],
            "par wszystkie": allan["pairs"],
        }
    ).round(2)
)

allan_figure(
    {
        "wszystkie fragmenty": allan,
        "noc (22-10 wg zegara RPi)": night,
        "dzień (13-20 wg zegara RPi)": afternoon,
        "biały szum, ta sama σ(1 s)": white,
    },
    FIGURE_WIDTH,
).show()

### Osobno dla każdego fragmentu

Krzywa każdego fragmentu, kolor = udział próbek z dnia (`DAY_HOURS` wg zegara RPi), przerywana = biały szum z poprzedniej komórki. Punkt τ liczony tylko, gdy fragment mieści co najmniej `MIN_ALLAN_AVERAGES` odcinków długości τ (krótsze dają niewiarygodne wartości). Po co: łączna krzywa jest średnią kwadratów, więc dominują w niej najbardziej zmienne i najdłuższe fragmenty; osobne krzywe pokazują, czy fragmenty dzielą się na spokojne (spadek jak biały szum, potem minimum) i niespokojne (wzrost od początku), i przy jakim τ spokojne mają minimum — to czas uśredniania, powyżej którego dłuższe uśrednianie przestaje pomagać.

W tabeli: `min_allan_kg` i `tau_at_min_s` — najmniejsza wartość krzywej i τ, przy którym występuje; `min_to_1s` — ile razy minimum jest mniejsze od σ(1 s) (1 = uśrednianie nic nie daje).

In [ ]:
per_fragment = fragment_allan(samples, fragments, DAY_HOURS)
display(collapsible(per_fragment.sort_values("min_to_1s").round(2), "Odchylenie Allana fragmentów"))

fragment_allan_figure(per_fragment, white, FIGURE_WIDTH).show()